# Advanced Production RAG

Production retrieval-augmented generation is a pipeline that finds evidence, ranks it, builds a focused context, and produces an answer grounded in that evidence.


## What to learn

- Hybrid retrieval combines keyword and semantic search.
- Reranking applies a stronger relevance model to a small candidate set.
- Metadata filters enforce scope such as tenant, date, or document type.
- Evaluation measures retrieval quality separately from answer quality.

## Decision rule

Add complexity only for a measured failure: hybrid search for missed exact terms, reranking for poor ordering, query rewriting for vocabulary mismatch, and graph retrieval for relationship-heavy questions.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Hybrid retrieval with RRF fusion and a toy reranker — the production
funnel in miniature. Structure identical to the real thing; swap the toy
scorers for BM25/pgvector/a cross-encoder.
"""
DOCS = {
    "d1": "Reset your password from the account settings page.",
    "d2": "Error E-4012 means the authentication token expired.",
    "d3": "Login failures are usually caused by expired sessions.",
    "d4": "Billing cycles renew on the first of each month.",
    "d5": "If you cannot sign in, your session token may have lapsed.",
}
QUERY = "user cannot log in, error E-4012"

# --- two retrievers with disjoint strengths -------------------------------
# Define the data shape and small operations before running them.
def lexical_rank(query, docs):
    """BM25 stand-in: exact token overlap. Nails 'E-4012'."""
    q = set(query.lower().replace(",", "").split())
    scores = {d: len(q & set(t.lower().replace(".", "").split())) for d, t in docs.items()}
    return [d for d, s in sorted(scores.items(), key=lambda x: -x[1]) if s > 0]

def dense_rank(query, docs):
    """Embedding stand-in: hand-set semantic ranking. Nails paraphrase
    ('cannot log in' ~ 'sign in', 'login failures') but misses 'E-4012'."""
    return ["d3", "d5", "d1", "d2"]

def rrf(rankings, k=60, weights=None):
    weights = weights or [1.0] * len(rankings)
    scores = {}
    for w, ranking in zip(weights, rankings):
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + w / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

lex, den = lexical_rank(QUERY, DOCS), dense_rank(QUERY, DOCS)
fused = rrf([lex, den])
print("lexical :", lex)
print("dense   :", den)
print("RRF     :", [(d, round(s, 4)) for d, s in fused])
# d2 (exact error code) and d3/d5 (paraphrase) all surface — consensus wins.

# --- rerank stage: expensive scorer on the small fused candidate set ------
def rerank(query, candidates, top_n=2):
    """Cross-encoder stand-in: joint relevance judgment on [query, doc]."""
    relevance = {"d2": 0.95, "d5": 0.80, "d3": 0.75, "d1": 0.40, "d4": 0.05}
    return sorted(candidates, key=lambda d: -relevance[d])[:top_n]

final = rerank(QUERY, [d for d, _ in fused[:4]])
print("rerank  :", final, "-> sent to the LLM with citations")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
